# Construindo banco de dados das amostras de vigas

In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
import itertools
import numpy as np
import openpyxl
from parepy_toolbox import sampling_algorithm_structural_analysis
from obj import momento_limite_armadura_simples, area_aco_flexao_simples, momento_resistente_secao_sem_cor, obj_mestrado_victor

# Planejamento experimental

In [2]:
# Definindo os níveis das variáveis
h_levels = np.linspace(0.30, 0.70, 2)
b_levels = np.linspace(0.14, 0.35, 2)
f_levels = np.linspace(20000, 50000, 2)
divs = 2 # Levels for mechanical steel rate

# Gerando o planejamento fatorial completo com itertools.product
combinacoes = list(itertools.product(h_levels, b_levels, f_levels))

# Convertendo para DataFrame
df = pd.DataFrame(combinacoes, columns=['h', 'b_w', 'f_ck'])
df

,h,b_w,f_ck
0,0.3,0.14,20000.0
1,0.3,0.14,50000.0
2,0.3,0.35,20000.0
3,0.3,0.35,50000.0
4,0.7,0.14,20000.0
5,0.7,0.14,50000.0
6,0.7,0.35,20000.0
7,0.7,0.35,50000.0


# Criando o banco de armaduras

In [3]:
b_w = []
h = []
f_ck = []
m_rdlim = []
pho_s = []
a_s = []
for i in range(len(df)):
    b_w.append(df['b_w'][i])
    h.append(df['h'][i])
    f_ck.append(df['f_ck'][i])
    m_rdlim.append(momento_limite_armadura_simples(df['b_w'][i], df['h'][i], df['f_ck'][i]))
    a_saux, pho_saux = area_aco_flexao_simples(m_rdlim[i], df['b_w'][i], df['h'][i], df['f_ck'][i])
    pho_s.append(pho_saux)
    a_s.append(a_saux)

df['m_rdlim'] = m_rdlim
df['a_s'] = a_s
df['pho_s'] = pho_s
df

,h,b_w,f_ck,m_rdlim,a_s,pho_s
0,0.3,0.14,20000.0,36.584136,0.000380,0.904886
1,0.3,0.14,50000.0,91.460340,0.000950,2.262214
2,0.3,0.35,20000.0,91.460340,0.000950,0.904886
3,0.3,0.35,50000.0,228.650850,0.002375,2.262214
4,0.7,0.14,20000.0,199.180296,0.000887,0.904886
5,0.7,0.14,50000.0,497.950740,0.002217,2.262214
6,0.7,0.35,20000.0,497.950740,0.002217,0.904886
7,0.7,0.35,50000.0,1244.876850,0.005542,2.262214


# Subdividindo em grupos por classe de densidade de armadura

In [4]:
b_w = []
h = []
f_ck = []
m_rdlim = []
pho_s = []
a_s = []
for indece, linha in df.iterrows():
    aux = linha['pho_s'] / divs
    pho = 0
    for i in range(divs):
        pho += aux
        pho_s.append(pho)
        b_w.append(linha['b_w'])
        h.append(linha['h'])
        f_ck.append(linha['f_ck'])
        m_rdlim.append(linha['m_rdlim'])
        a_s.append(pho * linha['b_w'] * linha['h'] / 100)

df_aux = {'b_w': b_w, 'h': h, 'f_ck': f_ck, 'm_rdlim': m_rdlim, 'a_s': a_s, 'pho_s': pho_s}
df_aux = pd.DataFrame(df_aux)
df_aux['m_rd'] = df_aux.apply(lambda row: momento_resistente_secao_sem_cor(row['a_s'], row['b_w'], row['h'],row['f_ck']), axis=1)
df_aux

,b_w,h,f_ck,m_rdlim,a_s,pho_s,m_rd
0,0.14,0.3,20000.0,36.584136,0.000190,0.452443,22.998372
1,0.14,0.3,20000.0,36.584136,0.000380,0.904886,40.686467
2,0.14,0.3,50000.0,91.460340,0.000475,1.131107,57.495929
3,0.14,0.3,50000.0,91.460340,0.000950,2.262214,101.716167
4,0.35,0.3,20000.0,91.460340,0.000475,0.452443,57.495929
5,0.35,0.3,20000.0,91.460340,0.000950,0.904886,101.716167
6,0.35,0.3,50000.0,228.650850,0.001188,1.131107,143.739823
7,0.35,0.3,50000.0,228.650850,0.002375,2.262214,254.290418
8,0.14,0.7,20000.0,199.180296,0.000443,0.452443,125.213357
9,0.14,0.7,20000.0,199.180296,0.000887,0.904886,221.515208


In [5]:
df_aux.drop(['m_rdlim'], axis='columns', inplace=True)
df_aux

,b_w,h,f_ck,a_s,pho_s,m_rd
0,0.14,0.3,20000.0,0.000190,0.452443,22.998372
1,0.14,0.3,20000.0,0.000380,0.904886,40.686467
2,0.14,0.3,50000.0,0.000475,1.131107,57.495929
3,0.14,0.3,50000.0,0.000950,2.262214,101.716167
4,0.35,0.3,20000.0,0.000475,0.452443,57.495929
5,0.35,0.3,20000.0,0.000950,0.904886,101.716167
6,0.35,0.3,50000.0,0.001188,1.131107,143.739823
7,0.35,0.3,50000.0,0.002375,2.262214,254.290418
8,0.14,0.7,20000.0,0.000443,0.452443,125.213357
9,0.14,0.7,20000.0,0.000887,0.904886,221.515208


# Análise de confiabilidade no tempo

In [6]:
tempos = list(range(0, 101, 20))
beta = []
b_w_aux_list = []
h_aux_list = []
f_ck_aux_list = []
m_rd_aux_list = []
a_s_aux_list = []
mgk_aux_list = []
mqk_aux_list = []
time_aux_list = []
for i, row in df_aux.iterrows():
    b_w_aux = row['b_w']
    h_aux = row['h']
    f_ck_aux = row['f_ck']
    m_rd_aux = row['m_rd']
    a_s_aux = row['a_s']
    chi_list = list(np.linspace(0.15, 0.65, divs, endpoint=True))
    gamma_g = 1.40
    gamma_q = 1.40
    dados_viga = {'h (m)': h_aux, 'b_w (m)': b_w_aux, 'm_rd (kN.m)': m_rd_aux, 'a_s (m2)': a_s_aux, 'gamma_c': 1.00, 'gamma_s': 1.00, 'gamma_f': 1.00}
    none_variable = {'dados_viga': dados_viga, 'time analysis': tempos}

    for id, chi in enumerate(chi_list):
        den_g = gamma_g + gamma_q*chi/(1-chi)
        den_q = gamma_g*(1-chi)/chi + gamma_q
        m_gk = m_rd_aux/den_g
        m_qk = m_rd_aux/den_q

        # Data
        g = {'type': 'normal', 'parameters': {'mean': 1.06*m_gk, 'sigma': 0.12*1.06*m_gk}, 'stochastic variable': False}
        q = {'type': 'gumbel max', 'parameters': {'mean': 0.21*m_qk, 'sigma': 0.21*0.76*m_qk}, 'stochastic variable': True}
        f_ck = {'type': 'normal', 'parameters': {'mean': 1.22*f_ck_aux, 'sigma': 0.15*1.22*f_ck_aux}, 'stochastic variable': False}
        f_yk = {'type': 'normal', 'parameters': {'mean': 1.22*500000, 'sigma': 0.04*1.22*500000}, 'stochastic variable': False}
        teta_r = {'type': 'normal', 'parameters': {'mean': 1, 'sigma': 0.05}, 'stochastic variable': False}
        teta_s = {'type': 'normal', 'parameters': {'mean': 1, 'sigma': 0.05}, 'stochastic variable': False}
        var = [g, q, f_ck, f_yk, teta_r, teta_s]

        # PAREpy setup
        setup = {
                    'number of samples': 2500,
                    'numerical model': {'model sampling': 'mcs-time', 'time steps': len(none_variable['time analysis'])},
                    'variables settings': var,
                    'number of state limit functions or constraints': 1,
                    'none variable': none_variable,
                    'objective function': obj_mestrado_victor,
                    'name simulation': 'victor',
                }
        # Call algorithm
        results_aux, pf_aux, beta_aux = sampling_algorithm_structural_analysis(setup)
        pf_aux_aux = pf_aux['G_0'].tolist() # A função pf vira uma lista
        time_limit = -100 # Aqui vcs devem colocar a função de vcs
        b_w_aux_list.append(b_w_aux)
        h_aux_list.append(h_aux)
        f_ck_aux_list.append(f_ck_aux)
        m_rd_aux_list.append(m_rd_aux)
        mgk_aux_list.append(m_gk)
        mqk_aux_list.append(m_qk)
        a_s_aux_list.append(a_s_aux)
        time_aux_list.append(time_limit)
        
        # Supondo que você tenha um DataFrame chamado df_aux
        #results_aux.to_excel("df_aux.xlsx", index=False)

final_df = {
            'b_w': b_w_aux_list,
            'h': h_aux_list,
            'f_ck': f_ck_aux_list,
            'm_rd': m_rd_aux_list,
            'm_gk': mgk_aux_list,
            'm_qk': mqk_aux_list,
            'a_s': a_s_aux_list,
            'time': time_aux_list,
           }
final_df = pd.DataFrame(final_df)


11:23:41 - Checking inputs completed!
11:23:41 - Started State Limit Function evaluation (g)...
11:23:43 - Finished State Limit Function evaluation (g) in 2.61e+00 seconds!
11:23:43 - Started evaluation beta reliability index and failure probability...
11:23:43 - Finished evaluation beta reliability index and failure probability in 6.59e-02 seconds!
11:23:44 - Voilà!!!!....simulation results are saved in victor_MCS-TIME_20250319-112343.txt
11:23:44 - Checking inputs completed!
11:23:44 - Started State Limit Function evaluation (g)...
11:23:46 - Finished State Limit Function evaluation (g) in 2.62e+00 seconds!
11:23:46 - Started evaluation beta reliability index and failure probability...
11:23:46 - Finished evaluation beta reliability index and failure probability in 9.90e-02 seconds!
11:23:46 - Voilà!!!!....simulation results are saved in victor_MCS-TIME_20250319-112346.txt
11:23:46 - Checking inputs completed!
11:23:46 - Started State Limit Function evaluation (g)...
11:23:49 - Finis

In [7]:
final_df.to_excel("final_df_tempo_limite.xlsx", index=False)

In [8]:
final_df.head(20)

,b_w,h,f_ck,m_rd,m_gk,m_qk,a_s,time
0,0.14,0.3,20000.0,22.998372,13.963297,2.464111,0.000190,-100
1,0.14,0.3,20000.0,22.998372,5.749593,10.677815,0.000190,-100
2,0.14,0.3,20000.0,40.686467,24.702498,4.359264,0.000380,-100
3,0.14,0.3,20000.0,40.686467,10.171617,18.890145,0.000380,-100
4,0.14,0.3,50000.0,57.495929,34.908243,6.160278,0.000475,-100
5,0.14,0.3,50000.0,57.495929,14.373982,26.694539,0.000475,-100
6,0.14,0.3,50000.0,101.716167,61.756244,10.898161,0.000950,-100
7,0.14,0.3,50000.0,101.716167,25.429042,47.225363,0.000950,-100
8,0.35,0.3,20000.0,57.495929,34.908243,6.160278,0.000475,-100
9,0.35,0.3,20000.0,57.495929,14.373982,26.694539,0.000475,-100
